# Ridge IC Native Adapter v1

目标：把 `ridge_ic` 从 proxy feature adapter 推进为 native IC baseline。该模型使用 rank-normalized forward return 目标训练 Ridge，并在 estimator 中保留 IC 目标元数据；可复用训练、回测、analysis 和 manifest 逻辑全部调用 `src/`。

## 外部参考边界

- scikit-learn Ridge API: `https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html`，核对 Ridge L2 线性回归 baseline 语义。
- qlib analysis/report 文档: `https://qlib.readthedocs.io/en/latest/component/report.html`，确认 IC/rank IC 属于预测排序质量和分析评估范畴。

本实验不复制第三方代码；只把本仓库已有 IC/ranking objective 明确为 native estimator contract。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import RidgeICRegressor, fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'ridge_ic_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ('ridge_ic', 'ridge_mse', 'hist_gradient_mse', 'xgboost_mse', 'sequence_lag_ridge_mse')
outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=MODEL_NAMES,
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:6])
estimator, constant_prediction = fit_estimator_for_model('ridge_ic', train=train, feature_columns=feature_columns)

capabilities = model_capability_table()
ridge_capability = capabilities.loc[capabilities['model'].eq('ridge_ic')].iloc[0].to_dict()
full_fidelity = model_fidelity_summary()
matrix_fidelity = outputs.summary['model_fidelity']

assert constant_prediction is None
assert isinstance(estimator, RidgeICRegressor)
assert model_adapter_name('ridge_ic') == 'ridge_ic_rank_correlation_v1'
assert estimator.objective_name_ == 'ic'
assert estimator.target_kind_ == 'forward_return_rank'
assert abs(estimator.rank_target_mean_) < 1e-12
assert estimator.rank_target_std_ > 0.0
assert ridge_capability['paper_fidelity'] == 'native_library'
assert ridge_capability['is_proxy'] is False
assert full_fidelity['fidelity_counts']['proxy_feature_adapter'] == 1
assert 'ridge_ic' in full_fidelity['native_models']
assert 'ridge_ic' not in full_fidelity['proxy_models']
assert full_fidelity['proxy_models'] == ['extra_trees_rank']
assert matrix_fidelity['proxy_models'] == []
assert 'ridge_ic' in matrix_fidelity['native_models']

notebook_summary = {
    'schema_version': 'research.ridge_ic_native_adapter_v1',
    'adapter_name': model_adapter_name('ridge_ic'),
    'estimator_class': estimator.__class__.__name__,
    'objective_name': estimator.objective_name_,
    'target_kind': estimator.target_kind_,
    'rank_target_mean': estimator.rank_target_mean_,
    'rank_target_std': estimator.rank_target_std_,
    'ridge_ic_capability': ridge_capability,
    'full_model_fidelity': full_fidelity,
    'matrix_model_fidelity': matrix_fidelity,
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
    'daily_backtest': str((CARD_DIR / 'daily_backtest.csv').relative_to(REPO_ROOT)),
    'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
    'analysis_manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
    'remaining_proxy_models': full_fidelity['proxy_models'],
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(notebook_summary, ensure_ascii=False, indent=2), encoding='utf-8')
notebook_summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.ridge_ic_native_adapter_v1',
 'adapter_name': 'ridge_ic_rank_correlation_v1',
 'estimator_class': 'RidgeICRegressor',
 'objective_name': 'ic',
 'target_kind': 'forward_return_rank',
 'rank_target_mean': -2.3693784062122245e-18,
 'rank_target_std': 1.0,
 'ridge_ic_capability': {'model': 'ridge_ic',
  'family': 'linear',
  'objective': 'ic',
  'input_kind': 'tabular',
  'estimator': 'RidgeICRegressor',
  'paper_fidelity': 'native_library',
  'is_proxy': False,
  'implementation_notes': 'Uses the native RidgeICRegressor training path behind the QuantBench-style registry contract.',
  'promotion_blocker': '',
  'trade_instrument': 'SOXL',
  'support_universe': 'SOXL,SOXX,VIX,QQQ,SPY'},
 'full_model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 32,
  'fidelity_counts': {'native_graph': 9,
   'native_library': 9,
   'native_sequence': 7,
   'native_torch': 6,
   'proxy_feature_adapter': 1},
  'proxy_models': ['extra_trees_rank'],
  'native_m

## 结构说明

- `ridge_ic` 是 native IC baseline：输入是 tabular feature matrix，目标是居中的 rank-normalized forward return。
- estimator 记录 `objective_name_`、`target_kind_`、`rank_target_mean_`、`rank_target_std_`、`feature_count_`，证明训练路径使用显式 IC contract，而不是泛化 proxy adapter name。
- 该模型用于 IC/rank objective 的线性基准和消融，不声称替代复杂 ranking model。

## 中文结论

- 做了什么：新增 `RidgeICRegressor`，把 `ridge_ic` 从 proxy fidelity 升级为 `native_library`，adapter name 固定为 `ridge_ic_rank_correlation_v1`。
- 核心指标：本 notebook 训练并回测 `ridge_ic`、`ridge_mse`、`hist_gradient_mse`、`xgboost_mse`、`sequence_lag_ridge_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=31、proxy=1；剩余 proxy 为 `extra_trees_rank`。
- Regime/区间差异：本轮继续复用 `src/analysis` 的 window/regime/qlib 风险表，具体强弱分段以 `matrix_card/model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 `model_review_briefing.csv` 为准。
- 是否晋升：否。本次是模型实现和链路覆盖升级，不直接晋升生产策略。
- 风险：IC rank target 是线性 baseline，仍需在更长 walk-forward、压力窗口、交易成本和组合互补性下复核。
- 下一步：剩余 proxy 为 `extra_trees_rank`，需要围绕 ranking objective 明确真实优化目标后替换。